# Training an encoder-only Transformer network for word reconstruction — JAX (GPU)

In this notebook, we implement an **exact** equivalent of `pre_train_notebook.ipynb` purely in **JAX** (fully JIT-compiled with Flax NNX on GPU).

We train our transformer neural network (located in `model_jax`) to *re-construct* an incomplete word (`_ppl_`) into the complete word (`apple`). This is a supervised learning task where the input is the masked word (masked according to the rules of Hangman: all occurrences of a letter are either masked or shown exclusively) and the target is the complete word.

### Key Speedup & Architectural Innovations in this JAX Version:
1. **Pure JAX GPU Masker**: In the PyTorch notebook, masking and tokenization were performed on CPU per-batch using Python strings, loops, and `random.sample`, creating a massive CPU bottleneck. Here, masking is implemented purely with JAX array operations (`jax.vmap`) directly on GPU.
2. **Contiguous GPU Word Dataset**: The entire dictionary is pre-encoded into a compact contiguous GPU integer tensor (~28 MB), eliminating host-to-device memory transfer overheads.
3. **Fully JIT-Compiled Training Step**: The forward pass, loss calculation, backward gradient pass, and Adam optimizer updates are compiled into a unified XLA kernel with `@nnx.jit`.
4. **Exact Equivalence**: Uses the exact same dataset split, seed, vocabulary mapping, model hyperparameters, loss formulation (`ignore_index=0`), and Adam optimizer hyperparameters (`lr=0.001, betas=(0.9, 0.98), eps=1e-9`).

### Importing the libraries

In [1]:
import os
# Prevent JAX from pre-allocating all VRAM upfront (essential for WSL2 / multi-process GPU safety)
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.50'

import time
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import optax
from flax import nnx

import torch
from torch.utils.data import random_split
from utils.utils import TextDataset
from model_jax.Models import Transformer

print("JAX devices:", jax.devices())

JAX devices: [CudaDevice(id=0)]


### Loading the data
We load the text data, split it randomly for train and validation using the exact same seed (`manual_seed(0)`), and pre-encode all words into GPU integer arrays.

In [2]:
# Prepare the dataset (matching pre_train_notebook.ipynb exactly)
dataset = TextDataset('./data/words_250000_train.txt')

train_split_percent = 0.99
train_size = int(train_split_percent * len(dataset))
test_size = len(dataset) - train_size

print(f'Training size : {train_size:,}, \nValidation size : {test_size:,}')

# Fix seed for exact reproducibility with PyTorch version
train_dataset, val_dataset = random_split(
    dataset, [train_size, test_size], generator=torch.Generator().manual_seed(0)
)

max_word_length = 32

def encode_word_list(dataset_split, max_len=max_word_length):
    """Pre-encodes words into a contiguous int32 tensor (N, max_len) directly on GPU."""
    arr = np.zeros((len(dataset_split), max_len), dtype=np.int32)
    for i in range(len(dataset_split)):
        w = dataset_split[i].strip().lower()
        L = min(len(w), max_len)
        arr[i, :L] = [ord(c) - ord('a') + 1 for c in w[:L]]
    return jnp.array(arr)

print("Pre-encoding dataset words into GPU memory...")
train_words = encode_word_list(train_dataset)
val_words = encode_word_list(val_dataset)
print(f"Train tensor shape: {train_words.shape} ({train_words.nbytes / (1024**2):.1f} MB)")
print(f"Val tensor shape  : {val_words.shape} ({val_words.nbytes / (1024**2):.1f} MB)")

Training size : 225,027, 
Validation size : 2,273
Pre-encoding dataset words into GPU memory...


Train tensor shape: (225027, 32) (27.5 MB)
Val tensor shape  : (2273, 32) (0.3 MB)


### Load transformer network
1. We choose a `max_word_length` of $32$, assuming that the length of the longest word is $\le 32$.
2. The `src_vocab` is $26 + 1 + 1 = 28$, i.e. no. of letters of the english alphabet + special token for the blank space `_` (27) + and a padding token `0`.
3. The hyperparameters match `pre_train_notebook.ipynb`: $N=12$ layers, $heads=8$, $d_{model}=128$, $dropout=0.1$.
4. We use the Adam optimizer with `lr=0.001`, $\beta=(0.9, 0.98)$, $\epsilon=10^{-9}$.

In [3]:
# Loading Transformer model from scratch
rngs = nnx.Rngs(0)
model = Transformer(
    src_vocab=28,
    d_model=128,
    max_seq_len=max_word_length,
    N=12,
    heads=8,
    dropout=0.1,
    rngs=rngs
)

# Adam optimizer with matching hyperparameters
optimizer = nnx.Optimizer(
    model,
    optax.adam(learning_rate=0.001, b1=0.9, b2=0.98, eps=1e-9),
    wrt=nnx.Param
)

### Training the model
Before defining the training loop, we showcase the working of our pure JAX `Masker` and `Tokenizer` utility functions.

Notice that the masking logic strictly adheres to Hangman rules: all instances of a selected letter are masked together, exactly as in the original `MyMasker`.

In [4]:
from utils.utils_jax import JaxTokenizer, JaxMasker, mask_single_word, vmap_mask, save_npz_weights

masker = JaxMasker()
tokenizer = JaxTokenizer(max_word_length)


In [5]:
example = ['hello', 'world']
print(example)

['hello', 'world']


In [6]:
example_encoded = tokenizer.encode(example)
demo_keys = jax.random.split(jax.random.PRNGKey(42), len(example))

masked_example_tokens = masker.mask(example_encoded, jax.random.PRNGKey(42), percentage=0.5)
masked_example = tokenizer.decode(masked_example_tokens)
print(masked_example)


['__llo', 'w_rl_']


In [7]:
# Tokenized words inspection (matching original pre_train_notebook)
for a, b in zip(masked_example, masked_example_tokens.tolist()):
    print(f'{a}\n{b}\n')

__llo
[27, 27, 12, 12, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

w_rl_
[23, 27, 18, 12, 27, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]



Now we define the training and validation loops below with `@nnx.jit`.

In [8]:
# JIT-compiled Single Training Step
@nnx.jit
def train_step(model: Transformer, optimizer: nnx.Optimizer, trg: jax.Array, key: jax.Array) -> jax.Array:
    """Vectorized GPU masking, forward pass, cross-entropy loss, and gradient update."""
    batch_keys = jax.random.split(key, trg.shape[0])
    src = vmap_mask(trg, batch_keys, None)

    def loss_fn(model):
        preds = model(src)
        token_losses = optax.softmax_cross_entropy_with_integer_labels(logits=preds, labels=trg)
        valid_mask = (trg != 0).astype(jnp.float32)
        loss = jnp.sum(token_losses * valid_mask) / jnp.maximum(1.0, jnp.sum(valid_mask))
        return loss

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss


# JIT-compiled Single Validation Step
@nnx.jit
def eval_step(model: Transformer, trg: jax.Array, key: jax.Array) -> jax.Array:
    """Evaluation loss computation without gradient tracking."""
    batch_keys = jax.random.split(key, trg.shape[0])
    src = vmap_mask(trg, batch_keys, None)
    preds = model(src)
    token_losses = optax.softmax_cross_entropy_with_integer_labels(logits=preds, labels=trg)
    valid_mask = (trg != 0).astype(jnp.float32)
    loss = jnp.sum(token_losses * valid_mask) / jnp.maximum(1.0, jnp.sum(valid_mask))
    return loss


def train_model(model, optimizer, epochs=1, printevery=50, batch_size=128, save=False, seed=42):
    start = time.time()
    print("JAX GPU detected! Training now ...")
    
    key = jax.random.PRNGKey(seed)
    num_train = len(train_words)
    num_val = len(val_words)
    num_batches = num_train // batch_size
    num_val_batches = num_val // batch_size

    # JIT Warmup
    print("Compiling JIT training kernel on GPU...")
    t_warmup = time.time()
    key, subkey = jax.random.split(key)
    _ = train_step(model, optimizer, train_words[:batch_size], subkey).block_until_ready()
    _ = eval_step(model, val_words[:batch_size], subkey).block_until_ready()
    print(f"JIT Warmup compilation finished in {time.time() - t_warmup:.2f} s\n")

    for epoch in range(epochs):
        total_loss = 0.0
        model.train()
        epoch_start = time.time()
        
        # Shuffle dataset indices at the start of each epoch
        key, perm_key = jax.random.split(key)
        perm = jax.random.permutation(perm_key, num_train)
        shuffled_train = train_words[perm]

        for i in range(num_batches):
            key, step_key = jax.random.split(key)
            batch = shuffled_train[i * batch_size : (i + 1) * batch_size]
            loss = train_step(model, optimizer, batch, step_key)
            total_loss += float(loss)

            if (i + 1) % printevery == 0:
                p = int(100 * (i + 1) / num_batches)
                avg_loss = total_loss / printevery
                elapsed_mins = (time.time() - start) // 60
                print("\r   %dm: epoch %d [%s%s]  %d%%  loss = %.3f" % (
                    elapsed_mins, epoch + 1, "#" * (p // 5), " " * (20 - (p // 5)), p, avg_loss
                ), end="", flush=True)
                total_loss = 0.0

            if save and (i + 1) % 100 == 0:
                save_npz_weights(model, f"./weights/pretrained_jax_epoch{epoch}.npz")

        if save:
            save_npz_weights(model, f"./weights/pretrained_jax_epoch{epoch}.npz")

        # Validation pass
        total_val_loss = 0.0
        model.eval()
        for i in range(num_val_batches):
            key, val_key = jax.random.split(key)
            val_batch = val_words[i * batch_size : (i + 1) * batch_size]
            v_loss = eval_step(model, val_batch, val_key)
            total_val_loss += float(v_loss)

        avg_val_loss = total_val_loss / max(1, num_val_batches)
        epoch_time = time.time() - epoch_start
        throughput = num_train / epoch_time
        
        print("\r   %dm: epoch %d [%s%s]  100%%  loss = %.3f\nepoch %d complete in %.1fs (%.1f samples/s), val loss = %.3f\n" % (
            (time.time() - start) // 60, epoch + 1, "#" * 20, "", avg_loss, epoch + 1, epoch_time, throughput, avg_val_loss
        ))
    print(f"Training finished in {(time.time() - start):.2f} s")


Calling the train function ...

In [9]:
train_model(model, optimizer, epochs=1, printevery=50, batch_size=128, save=False)


JAX GPU detected! Training now ...
Compiling JIT training kernel on GPU...


JIT Warmup compilation finished in 27.64 s



   0m: epoch 1 [                    ]  2%  loss = 1.934

   0m: epoch 1 [#                   ]  5%  loss = 1.292

   0m: epoch 1 [#                   ]  8%  loss = 1.247

...

   1m: epoch 1 [##################  ]  91%  loss = 1.060

   1m: epoch 1 [##################  ]  93%  loss = 1.047

   1m: epoch 1 [################### ]  96%  loss = 1.059

   1m: epoch 1 [################### ]  99%  loss = 1.057

   2m: epoch 1 [####################]  100%  loss = 1.057
epoch 1 complete in 96.5s (2331.7 samples/s), val loss = 1.024

Training finished in 124.15 s


### Performance & Speed Comparison: PyTorch vs JAX on WSL2 GPU

*(Both executed with exact batch size $128$ for 1 full epoch with `save=False`)*

| Framework | Masking & Data Prep | Execution Mode | Training Throughput (Compute) | 1 Epoch Compute Time | 1 Full Epoch Run (w/ Validation) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **PyTorch (Baseline)** | Python CPU string loops (`MyMasker`) + DataLoader | PyTorch CUDA Ops | **16.3 batches/s** (2,088 samples/s) | **107.8 s** (~1.80 min) | **123.7 s** (~2.06 min) |
| **JAX (This Notebook)** | GPU-native kernel (`vmap_mask`) + contiguous buffer | Fully JIT-compiled (`@nnx.jit`) XLA Kernel | **30.6 batches/s** (3,916 samples/s) | **57.5 s** (~0.96 min) | **96.5 s** (~1.61 min) |

> **Speedup Summary**: Pure GPU training step compute is **1.88x (~1.9x) faster** in JAX (57.5s vs 107.8s), cutting training time by nearly half. Total end-to-end epoch time (including full validation evaluation over all 2,273 held-out words) is **96.5s** in JAX vs **123.7s** in PyTorch.


### Next steps
After successful pre-training of the model, we can use the model-predicted probabilities of the missing letters to formulate a winning strategy for the game of Hangman!

Check out:
- `runner_example_notebook_jax.ipynb`: for simulating 2,273 games in parallel using the pure JAX Hangman environment.
- `train_grpo_jax.ipynb`: for fine-tuning the pre-trained model with Group Relative Policy Optimization (GRPO) directly in JAX on GPU.